# Verilog<->AIG LLM-based tagging

In [1]:
from setup import setup_tools, aag_path, verilog_path

setup_tools()

Setting project root: /home/krishnendu/Research/fv-invariant-mining
Updated PATH to include: [PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadical/build'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/abc'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/oss-cad-suite/bin'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/aiger'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadiback')]


In [2]:
from aig_grapher import AIG

In [3]:
from typing import List, Dict, Any
from pathlib import Path

class AIG_data:
	def __init__(self, aig: AIG):
		self.aig = aig
		self.num_nodes = len(aig.nodes)
		self.data: List[Dict[str, str]] = [{} for _ in range(self.num_nodes)]

	def populate_data(self, indices: List[int], key: str, values: List[str]):
		assert len(indices) == len(values), "Length of indices and values must be the same."
		
		for idx, val in zip(indices, values):
			self.data[idx][key] = val

In [4]:
from ollama import Client
import os

os.environ["OLLAMA_API_KEY"] = "24691ae7a5aa4008b278b5859c17e1ec.kxf5OcJXXzJ9Ies9xlkjJlRr"
client = Client(
    host="https://ollama.com",
	# host="http://kfed:11434",
    headers={'Authorization': 'Bearer ' + str(os.environ.get('OLLAMA_API_KEY'))}
)

model_name = 'nemotron-3-super:cloud'
# model_name = "qwen3-coder:30b"
def query_llm(prompt: str):
	messages = [
	{
		'role': 'formal-verification-engineer',
		'content': prompt,
	},
	]

	res = client.chat(model_name, messages=messages, stream=False)
	return res


In [8]:
from typing import Tuple

def encode_fanins(aig: AIG, AIG_data: AIG_data, node_id: int, data_key: str) -> str:
	fanins = aig.nodes[node_id].fanins
	encoded_fanins = []
	for fanin_id, is_inverted in fanins:
		fanin_str = f"{'INVERTED_' if is_inverted else ''}node_{fanin_id} [{AIG_data.data[fanin_id].get(data_key, 'unknown')}]"
		encoded_fanins.append(fanin_str)
	return ", ".join(encoded_fanins)

def upward_propagation(aig: AIG, aig_data: AIG_data, levels: List[List[int]], query_llm: Any, verilog_embed):
	node_count = 0
	total_nodes = sum(len(level) for level in levels[1:])  # Exclude input level
	levels = levels[1:] # Skip the first level (inputs)
	for i, level in enumerate(levels):
		for node_id in level:
			prompts = []
			prompts.append("For a circuit whose condensed technical representation is given as follows:\n")
			prompts.append(verilog_embed)
			prompts.append(
				f", this is the analysis of an AIG node formed from it at level {i+1}. "
				"The node computes fanin[0] AND fanin[1]."
			)

			prompts.append(
				"Each fanin is encoded as: INVERTED_node_id [data]. "
				"'INVERTED_' means logical negation on that edge. "
				"'node_id' is only an identifier. "
				"'data' is the semantic summary of that fanin."
			)

			prompts.append(
				"Following is a compact, high-value semantic description of this node inferred from the fanin encodings and data for use in downstream verification tasks."
			)

			prompts.append(
				"The description does not contain trivial info like 'A AND B'. "
				"Instead captures likely intent, behavior, constraints, patterns, enable conditions, gating logic, mux-like structure, arithmetic role, comparison meaning, redundancy, or functional interpretation."
			)

			prompts.append(
				"Prefers abstractions that help equivalence checking across structurally different but functionally similar circuits."
			)

			prompts.append(
				"The description will be further used by another LLM, so it doesn't need to be human-friendly. It should be concise and information-dense, focusing on the most salient semantic aspects of the node's function."
			)

			prompts.append(
				"The very concise one line description and the last line of this message is as follows:"
			)

			encoded_fanins = encode_fanins(aig, aig_data, node_id, "upward_propagation")
			prompts.append(encoded_fanins)

			final_prompt = "\n".join(prompts)
			res = query_llm(final_prompt)
			# res = str(["test_data_for_node_" + str(node_id) for node_id in level])  # Mock response for testing

			res_data = res.message.content
			# write how many percentage of nodes have been processed out of total nodes for upward propagation
			print(f"{(node_count/total_nodes) * 100:.2%}: {res_data}")
			node_count += 1
			aig_data.populate_data([node_id], "upward_propagation", [res_data])

def build_aig_and_data(aig_path: Path, verilog_path: Path, top_module_name: str) -> Tuple[AIG, AIG_data, str]:
	print("Building internal AIG representation...")
	aig_obj = AIG(aig_path)
	aig_data = AIG_data(aig_obj)

	print("Reading Verilog file and generating condensed technical representation using LLM...")
	with open(verilog_path, 'r') as f:
		verilog_content = f.read()
		
	verilog_embed = query_llm(f"Analyze the {top_module_name} module in this verilog code and compile an extremely condensed technical representation for use in downstream LLM-based tasks. No human is going to read that representation. So, make it as much information dense as you want. Verilog code:\n{verilog_content}")
	verilog_embed = str(verilog_embed.message.content)
	print("Condensed technical representation generated.")

	aig_data.populate_data(aig_data.aig.input_ids, "upward_propagation", [f"input_bit_{i}" for i in range(len(aig_data.aig.input_ids))])
	aig_data.populate_data([0], "upward_propagation", ["constant_0/FALSE"])
	aig_data.populate_data([id for id, _ in aig_data.aig.outputs], "downward_propagation", [f"{'INVERTED_' if inv else ''}output_bit_{i}" for i, (_, inv) in enumerate(aig_data.aig.outputs)])

	aig_levels = aig_obj.bfs_levels()

	print("Starting upward propagation to infer semantic summaries for each node...")
	upward_propagation(aig_obj, aig_data, aig_levels, query_llm, verilog_embed)
	
	return aig_obj, aig_data, verilog_embed

In [9]:
A = build_aig_and_data(aag_path/"rc_addr_4bit.aag", verilog_path/"adders.v", "ripple_carry_adder")

Building internal AIG representation...
Reading Verilog file and generating condensed technical representation using LLM...
Condensed technical representation generated.
Starting upward propagation to infer semantic summaries for each node...
0.00%: INVERTED_node_5 [input_bit_4], INVERTED_node_1 [input_bit_0] -> Carry propagation gate: computes carry[5] as AND of inverted sum-bit-4 (indicating no generate) and inverted input-bit-0 (indicating no propagate from LSB), suppressing carry-in when neither bit generates nor propagates a carry from the LSB upward.
312.50%: carry[4] = a[0] & b[0] & ~carry[0] & (carry[0] | a[0] | b[0]) is equivalent to carry[4] = a[0] & b[0] since carry[0]=0. The node computes carry[4] from LSB stage; it is 1 iff a[0] and b[0] are both 1, i.e., generates a carry from bit 0 that propagates to bit 4 through zero stages — only possible if carry[0]=0 and all intermediate carries are 0, but since carry[i+1] depends only on bit i, carry[4] is 0 unless BIT_WIDTH<=4 and